# Cox Proportional Hazards

### Wstęp
&emsp;Model proporcjonalnego hazardu Cox'a jest modelem pozwalającym ocenić wpływ wielu czynników na funkcję hazardu (chwilowe natężenie ryzyka). W naszym przypadku oceniać będziemy wpływ na ryzyko zapłaty (zdarzenie).

&emsp; Przy budownie modelu uwzględnimy następujące cechy:
* `total_open_amount` *(kwota faktury)*
* `days_to_due` *(po ilu dniach płatność)*
* `invoice_age` *(wiek faktury)*
* `avg_delay_customer` *(średnie opóźnienie klienta)*
* `cust_payment_terms` *(warunki płatności)*
* `invoice_currency` *(waluta)*
* `segment` *(BE lub SME)*

### Biblioteki i przygotowanie danych
&emsp; Zacznijmy od zaimportowania niezbędnych narzędzi oraz przygotowania danych. Zostawimy jedynie kolumny odpowiadające interesującym nas cechom.

In [2]:
import pandas as pd
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

df = pd.read_csv('../data/dataset_survclean.csv')

features = [
    'total_open_amount',
    'days_to_due',
    'invoice_age',
    'avg_delay_customer',
    'cust_payment_terms',
    'invoice_currency',
    'segment',
    'time_days', 
    'event'
]

cox_df = df[features].copy()

cox_df = pd.get_dummies(cox_df, columns=['cust_payment_terms', 'invoice_currency', 'segment'], drop_first=True)
cox_df = cox_df.dropna()
cox_df = cox_df.astype(float)

display(cox_df.head())

,total_open_amount,days_to_due,invoice_age,avg_delay_customer,time_days,event,cust_payment_terms_NA10,cust_payment_terms_NA32,cust_payment_terms_NAA8,cust_payment_terms_NAAW,...,cust_payment_terms_NAVF,cust_payment_terms_NAVQ,cust_payment_terms_NAVR,cust_payment_terms_NAWN,cust_payment_terms_NAWP,cust_payment_terms_NAWU,cust_payment_terms_NAX2,cust_payment_terms_OTHER,invoice_currency_USD,segment_SME
0,54273.28,16.0,15.0,0.743688,17.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,79656.60,20.0,20.0,2.222222,17.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,2253.86,15.0,15.0,2.467532,107.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3299.70,11.0,10.0,6.260714,53.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,33133.29,15.0,15.0,0.743688,12.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Model
&emsp; Pozostało dopasować model Coxa do badanego zbioru danych. Kolumną zawierającą informacje na temat upływu czasu jest `time_days`, a kolumną informującą o wystąpieniu zdarzenia jest kolumna `event`.

In [3]:
cph = CoxPHFitter()

cph.fit(cox_df, duration_col='time_days', event_col='event')

<lifelines.CoxPHFitter: fitted with 48838 total observations, 9681 right-censored observations>

#### Załozenie o proporcjinalności hazardów
&emsp; Model Coxa opiera się na załozeniu o proporcjonalności hazardów. Załozenie to mowi, ze stosunek hazardu dla dowolnych dwoch obserwacji musi być stały w czasie. W kontekście projektu oznacza to, ze wpływ cechy na prawdopodobieństwo zapłacenia faktury nie moze się zmieniać wraz upływem dni. Wpływ ten musi być ponadto liniowy.

&emsp; Jeśli to załozenie jest łamane model traci na wiarygodności. W celu jego weryfikacji wykorzystamy test reszt Schoenfelda.

In [5]:
results = cph.check_assumptions(cox_df, p_value_threshold=0.05, show_plots=False)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'total_open_amount' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'total_open_amount' might be incorrect. That is,
there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'total_open_amount' using pd.cut, and then specify it in
`strata=['total_open_amount', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'days_to_due' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'days_to_due' might be incorrect. That is, there
may be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in lin

&emsp;Z tabeli odczytujemy, że dla wielu zmiennych (m.in. `avg_delay_customer`, `days_to_due`, `invoice_age`, `segment_SME`, `total_open_amount`), wartość p-value jest niższa niż próg 0.05. Oznacza to, że hipoteza zerowa testu Schoenfelda jest fałszywa. Co za tym idzie, wpływ cech na "ryzyko" opłacenia faktury nie jest stały w czasie.

&emsp;Mamy do czynienia ze złamaniem podstawowego założenia modelu Coxa, co pozwala wyciągnąć wniosek, że modele nieliniowe jak Random Survival Forest poradzą sobie z predykcją zauważalnie lepiej.

#### Ewaluacja modelu
&emsp; Bazując na budowie algorytmu do wyliczenia funkcji częściowej wiarygodności (partial likelihood), w której wzorze wspolczynnik dla kazdej cechy jest stały w czasie, otrzymane w tabeli wyniki mozemy interpretować jako uśredniowy wpływ poszczególnych cech na ryzyko zapłaty.

&emsp; Wywołanie `cph.print_summary` dostarczy nam informacji o m.in:
* Hazard Ratio (`exp(coef)`): 
* C-index (`Concordance`): 

In [4]:
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 48838 total observations, 9681 right-censored observations>
             duration col = 'time_days'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 48838
number of events observed = 39157
   partial log-likelihood = -393201.55
         time fit was run = 2026-07-30 15:39:25 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
total_open_amount         0.00      1.00      0.00            0.00            0.00                1.00                1.00
days_to_due              -0.04      0.97      0.00           -0.04           -0.03                0.96                0.97
invoice_age               0.02      1.02      0.00            0.01            0.02                1.01                1.02
avg_delay_customer       -0.02      0.98      0.00           -0.02           -0.02                0.98                0.98
cust_payment_terms_NA10   0.32      1.38      0.12            0.09            0.56                1.09                1.75
cust_payment_terms_NA32  -0.15      0.86      0.12           -0.37            0.08                0.69                1.08
cust_payment_terms_NAA8  -0.09      0.91      0.11           -0.30            0.11                0.74                1.12
cust_payment_terms_NAAW   0.25      1.29      0.13           -0.01            0.52                0.99                1.68
cust_payment_terms_NAAX  -0.00      1.00      0.11           -0.22            0.21                0.80                1.24
cust_payment_terms_NAC6  -0.14      0.87      0.11           -0.36            0.07                0.70                1.07
cust_payment_terms_NAD1  -0.16      0.85      0.11           -0.38            0.06                0.68                1.06
cust_payment_terms_NAD5  -0.16      0.85      0.12           -0.40            0.08                0.67                1.08
cust_payment_terms_NAG2  -0.09      0.91      0.11           -0.31            0.12                0.74                1.13
cust_payment_terms_NAGD   0.05      1.05      0.14           -0.22            0.32                0.80                1.37
cust_payment_terms_NAH4   0.21      1.23      0.11           -0.00            0.42                1.00                1.51
cust_payment_terms_NAM1   0.96      2.62      0.13            0.71            1.22                2.04                3.38
cust_payment_terms_NAM2   0.69      2.00      0.12            0.46            0.92                1.58                2.52
cust_payment_terms_NAM4   0.72      2.06      0.11            0.50            0.94                1.65                2.56
cust_payment_terms_NAU5  -0.09      0.91      0.11           -0.31            0.13                0.73                1.14
cust_payment_terms_NAVE  -0.25      0.78      0.13           -0.50           -0.00                0.61                1.00
cust_payment_terms_NAVF   0.03      1.03      0.14           -0.24            0.30                0.79                1.35
cust_payment_terms_NAVQ   0.12      1.13      0.16           -0.20            0.45                0.82                1.56
cust_payment_terms_NAVR   0.45      1.57      0.19            0.07            0.83                1.07                2.30
cust_payment_terms_NAWN   0.26      1.29      0.18           -0.09            0.60                0.92                1.83
cust_payment_terms_NAWP  -0.18      0.83      0.17           -0.51            0.15                0.60                1.16
cust_payment_terms_NAWU   0.12      1.13      0.15           -0.17            0.40                0.85                1.50
cust_payment_terms_NAX2  -0.02      0.98      0.13           -0.27            0.23                0.77                1.25
cust_payment_terms_OTHER  0.06      1.06      0.11           